### 1. Download/Load SP500 stocks prices data.

In [3]:
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')


In [4]:
import pandas as pd
import requests
import certifi
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, verify=certifi.where())
html = response.text

tables = pd.read_html(StringIO(html))

sp500 = tables[0]

sp500['Symbol'] = sp500["Symbol"].str.replace('.', '-')

symbols_list = sp500["Symbol"].unique().tolist()

end_date = '2026-03-20'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

df = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date,
    auto_adjust=False  # 👈 IMPORTANT
).stack()

df.index.names = ['date', 'ticker']

df.columns= df.columns.str.lower()

df

[*********************100%***********************]  503 of 503 completed

1 Failed download:
['WDAY']: TypeError("'NoneType' object is not subscriptable")


Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.545273   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667408   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621094   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924828   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.839996   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  
date       ticker               
2018-03-22 A         1690000.0  
           AAPL    165963200.0  
           ABBV     26861200.0  
           ABNB            NaN  
           ABT       5345300.0  
...                        ...  
2026-03-19 XYZ       6387300.0  
           YUM       1832100.0  
           ZBH       1812000.0  
           ZBRA       593300.0  
           ZTS       4999100.0  

[1010527 rows x 6 columns]

### 2. Calculate features and technical indicators for each stock.
- Garman-Klass Volatility
- RSI
- Bollinger Bands
- ATR
- MACD
- Dollar Volume

In [7]:
# Columns to ensure are numeric
cols_to_convert = ['open', 'high', 'low', 'close', 'adj close', 'volume']

df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

In [15]:
# Force all numeric columns to float first
numeric_cols = ['open', 'high', 'low', 'close', 'adj close', 'volume']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Garman-Klass Volatility
df['garman_klass_vol'] = (
    (np.log(df['high']) - np.log(df['low']))**2
) / 2 - (2 * np.log(2) - 1) * (
    np.log(df['adj close']) - np.log(df['open'])
)**2

# RSI
df['rsi'] = df.groupby(level=1)['adj close'].transform(
    lambda x: pandas_ta.rsi(close=x.astype(float), length=20)
)

# Bollinger Bands
def compute_bbands(x):
    bb = pandas_ta.bbands(close=np.log1p(x.astype(float)), length=20)
    if bb is None:
        return pd.DataFrame(np.nan, index=x.index, columns=[0, 1, 2])
    return bb

df['bb_low'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 0]
)
df['bb_mid'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 1]
)
df['bb_high'] = df.groupby(level=1)['adj close'].transform(
    lambda x: compute_bbands(x).iloc[:, 2]
)

# ATR
def compute_atr(stock_data):
    try:
        atr = pandas_ta.atr(
            high=stock_data['high'].astype(float),
            low=stock_data['low'].astype(float),
            close=stock_data['adj close'].astype(float),
            length=14
        )
        if atr is None or atr.isna().all():
            return pd.Series(np.nan, index=stock_data.index)
        atr = atr.sub(atr.mean()).div(atr.std())
        atr.index = stock_data.index
        return atr
    except Exception:
        return pd.Series(np.nan, index=stock_data.index)

df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# MACD
def compute_macd(close):
    try:
        macd = pandas_ta.macd(close=close.astype(float), length=20).iloc[:, 0]
        return macd.sub(macd.mean()).div(macd.std())
    except Exception:
        return pd.Series(np.nan, index=close.index)

df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

# Dollar Volume
df['dollar_volume'] = (df['adj close'] * df['volume']) / 1e6

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.545273   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667408   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621094   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924828   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.839996   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2018-03-22 A         1690000.0         -0.001975        NaN       NaN   
           AAPL    165963200.0         -0.001552        NaN       NaN   
           ABBV     26861200.0         -0.058747        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT       5345300.0         -0.009116        NaN       NaN   
...                        ...               ...        ...       ...   
2026-03-19 XYZ       6387300.0          0.000407  47.622919  3.927554   
           YUM       1832100.0          0.000139  45.700865  5.048697   
           ZBH       1812000.0          0.000212  42.287756  4.491567   
           ZBRA       593300.0          0.000443  37.090520  5.274545   
           ZTS       4999100.0          0.000207  39.486943  4.737676   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2018-03-22 A            NaN       NaN       NaN       NaN     107.391511  
           AAPL         NaN       NaN       NaN       NaN    6583.329966  
           ABBV         NaN       NaN       NaN       NaN    1870.106123  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     282.899081  
...                     ...       ...       ...       ...            ...  
2026-03-19 XYZ     4.109759  4.291963 -0.541365  0.056086     376.786838  
           YUM     5.090827  5.132957 -1.732438 -0.236914     286.265625  
           ZBH     4.567495  4.643423 -1.309945 -0.345068     162.790073  
           ZBRA    5.401648  5.528752  0.095328 -1.256195     122.332528  
           ZTS     4.821242  4.904809 -1.780040 -1.013602     579.845598  

[1010527 rows x 14 columns]

### 3. Aggregate to monthly level and filter top 150 most liquid stocks for each month
- To reduce training time and experiment with features and strategies, we convert the business-daily data to month-end frequency

In [16]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]

data = pd.concat([
    df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
    df.unstack()[last_cols].resample('ME').last().stack('ticker')
], axis=1).dropna(subset=['dollar_volume', 'adj close'])  # only drop if these are NaN

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-03-31 A          143.276082   63.008434         -0.001044   8.661196   
           AAPL      6347.315107   39.416027         -0.001092  10.343299   
           ABBV       956.063209   67.172661         -0.044829  14.218648   
           ABT        332.180025   52.047523         -0.006877   7.529500   
           ACGL        64.065371   27.129135         -0.001026  12.204337   
...                          ...         ...               ...        ...   
2026-03-31 XYZ        581.353228   58.990002          0.000407  47.622919   
           YUM        279.860158  156.250000          0.000139  45.700865   
           ZBH        200.072727   89.839996          0.000212  42.287756   
           ZBRA       162.709509  206.190002          0.000443  37.090520   
           ZTS        481.208041  115.989998          0.000207  39.486943   

                     bb_low    bb_mid   bb_high       atr      macd  
date       ticker                                                    
2018-03-31 A            NaN       NaN       NaN       NaN       NaN  
           AAPL         NaN       NaN       NaN       NaN       NaN  
           ABBV         NaN       NaN       NaN       NaN       NaN  
           ABT          NaN       NaN       NaN       NaN       NaN  
           ACGL         NaN       NaN       NaN       NaN       NaN  
...                     ...       ...       ...       ...       ...  
2026-03-31 XYZ     3.927554  4.109759  4.291963 -0.541365  0.056086  
           YUM     5.048697  5.090827  5.132957 -1.732438 -0.236914  
           ZBH     4.491567  4.567495  4.643423 -1.309945 -0.345068  
           ZBRA    5.274545  5.401648  5.528752  0.095328 -1.256195  
           ZTS     4.737676  4.821242  4.904809 -1.780040 -1.013602  

[47740 rows x 9 columns]

- Calculate 5-year rolling average of dollar volume for each stocks before filtering.

In [17]:
data['dollar_volume'] = (data.loc[:,'dollar_volume'].unstack('ticker').rolling(5*12).mean().stack('ticker')
    .reindex(data.index))

data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2023-02-28 AAPL    145.304947          0.000061  52.120108  4.963779   
           ABBV    138.085938         -0.004509  54.426377  4.861899   
           ABT      95.904297         -0.000260  36.762976  4.548466   
           ACN     254.677628         -0.000341  41.629989  5.532744   
           ADBE    323.950012          0.000107  37.957137  5.774364   
...                       ...               ...        ...       ...   
2026-03-31 WDC     316.929993          0.002351  62.481774  5.485216   
           WFC      76.389999          0.000358  35.566775  4.293305   
           WMT     120.841995          0.000350  45.694634  4.804058   
           XOM     158.160004          0.000451  65.453187  4.986162   
           XYZ      58.990002          0.000407  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2023-02-28 AAPL    5.006365  5.048952 -0.426912  0.337159  
           ABBV    4.907300  4.952702  0.036110 -0.065564  
           ABT     4.624205  4.699945 -0.300277 -1.836136  
           ACN     5.596135  5.659525 -0.506610 -0.658135  
           ADBE    5.896798  6.019232 -0.151346 -0.758460  
...                     ...       ...       ...       ...  
2026-03-31 WDC     5.622986  5.760756  5.844734  3.115020  
           WFC     4.400714  4.508124 -0.948356 -3.360090  
           WMT     4.835141  4.866224  0.219226 -0.703873  
           XOM     5.032413  5.078663 -1.321983  2.399187  
           XYZ     4.109759  4.291963 -0.541365  0.056086  

[5662 rows x 8 columns]

### 4. Calculate Monthly Returns for different time horizons as features.

- To capture time series dynamics that reflect, for example, momentum patterns, we compute historical returns using the method .pct_change(lag). returns over various monthly periods as identified by lags.

In [18]:
def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1,2,3,6,9,12]

    for lag in lags:
        df[f'return_{lag}m'] = (df['adj close']
                            .pct_change(lag)
                           .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                  upper=x.quantile(1-outlier_cutoff)))
                            .add(1)
                            .pow(1/lag)
                            .sub(1))
    return df

data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data



adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2024-02-29 AAPL    179.119843          0.000086  40.958208  5.184216   
           ABBV    164.349014         -0.002304  65.103500  5.065721   
           ABT     114.129013         -0.000949  63.371029  4.662014   
           ACN     364.986450         -0.000524  58.736124  5.863055   
           ADBE    560.280029          0.000086  42.704909  6.250733   
...                       ...               ...        ...       ...   
2026-03-31 VZ       49.480000          0.000385  58.273822  3.907932   
           WFC      76.389999          0.000358  35.566775  4.293305   
           WMT     120.841995          0.000350  45.694634  4.804058   
           XOM     158.160004          0.000451  65.453187  4.986162   
           XYZ      58.990002          0.000407  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2024-02-29 AAPL    5.215029  5.245841 -0.887072 -0.971514  -0.018543   
           ABBV    5.101149  5.136577 -0.631005  1.340015   0.070864   
           ABT     4.716174  4.770334 -0.789979  1.285698   0.048520   
           ACN     5.890862  5.918669 -0.549923  0.772604   0.029955   
           ADBE    6.372520  6.494307  1.086732 -1.500612  -0.093075   
...                     ...       ...       ...       ...        ...   
2026-03-31 VZ      3.937403  3.966873 -1.605598  2.016359  -0.013163   
           WFC     4.400714  4.508124 -0.948356 -3.360090  -0.062124   
           WMT     4.835141  4.866224  0.219226 -0.703873  -0.053615   
           XOM     5.032413  5.078663 -1.321983  2.399187   0.037115   
           XYZ     4.109759  4.291963 -0.541365  0.056086  -0.073940   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2024-02-29 AAPL    -0.030458  -0.015994  -0.005989   0.002614    0.017588  
           ABBV     0.070925   0.076702   0.033908   0.030947    0.014615  
           ABT      0.040705   0.045588   0.025772   0.018643    0.014604  
           ACN      0.035337   0.041304   0.026014   0.024111    0.028920  
           ADBE    -0.030917  -0.028479   0.000280   0.033144    0.044026  
...                      ...        ...        ...        ...         ...  
2026-03-31 VZ       0.054235   0.073154   0.025787   0.020699    0.012845  
           WFC     -0.078802  -0.060419  -0.013684  -0.003382    0.006989  
           WMT      0.008150   0.028162   0.027583   0.024539    0.027742  
           XOM      0.061124   0.097802   0.059397   0.046370    0.026927  
           XYZ     -0.011986  -0.032269  -0.033274  -0.015557    0.006881  

[3615 rows x 14 columns]

### 5. Download Fama-French Factors and Calculate Rolling Factor Betas.

- We will introduce the Fama-French data to estimate the exposure of assets to common risk factors using linear regressionn.
- The five Fama-French factors, namely market risk, size, value, operating profitability, and investment have been shown empirically to explain asset returns and are commonly used to assess the risk/return profile of portfolios. Hence, it is natural to include past factor exposures as financial features in models.
- We can access the historical factor returns using the pandas-datareader and estimate historical exposures using the RollingOLS rolling linear regression.

In [19]:
import zipfile, io, requests

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip"

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

ff_factors = pd.read_csv(
    z.open(z.namelist()[0]),
    skiprows=3,
    index_col=0
)

# The file has annual data appended at the bottom, cut it off
ff_factors = ff_factors[ff_factors.index.astype(str).str.strip().str.match(r'^\d{6}$')]

# Convert index to datetime to match your data's date index
ff_factors.index = pd.to_datetime(ff_factors.index, format='%Y%m') + pd.offsets.MonthEnd(0)

# Filter from 2010 onwards
ff_factors = ff_factors.loc['2010':]

# drop RF column and divide by 100
factor_data = ff_factors.apply(pd.to_numeric, errors='coerce').drop('RF', axis=1).div(100)

factor_data.index.name = 'date'

factor_data = factor_data.join(data['return_1m']).sort_index()

factor_data




Mkt-RF     SMB     HML     RMW     CMA  return_1m
date       ticker                                                   
2024-02-29 AAPL    0.0507 -0.0082 -0.0343 -0.0206 -0.0217  -0.018543
           ABBV    0.0507 -0.0082 -0.0343 -0.0206 -0.0217   0.070864
           ABT     0.0507 -0.0082 -0.0343 -0.0206 -0.0217   0.048520
           ACN     0.0507 -0.0082 -0.0343 -0.0206 -0.0217   0.029955
           ADBE    0.0507 -0.0082 -0.0343 -0.0206 -0.0217  -0.093075
...                   ...     ...     ...     ...     ...        ...
2026-01-31 WFC     0.0102  0.0326  0.0370  0.0183  0.0181  -0.029077
           WMT     0.0102  0.0326  0.0370  0.0183  0.0181   0.069383
           XOM     0.0102  0.0326  0.0370  0.0183  0.0181   0.163687
           XYZ     0.0102  0.0326  0.0370  0.0183  0.0181  -0.071593
           ZTS     0.0102  0.0326  0.0370  0.0183  0.0181  -0.003712

[3342 rows x 6 columns]

- Filter out stocks with less than 10 months of data.

In [20]:
factor_data.groupby(level=1).size()


ticker
AAPL    24
ABBV    24
ABT     24
ACN     24
ADBE    24
        ..
WFC     24
WMT     24
XOM     24
XYZ     24
ZTS     14
Length: 155, dtype: int64